# Clasificacion Cuantica sobre el Dataset Iris
## Comparacion de Modelos VQC y QSVC frente a Baselines Clasicos

**Universidad Nacional de San Antonio Abad del Cusco**  
Departamento Academico de Informatica — Computacion Cuantica

---

Se implementan dos modelos cuanticos (VQC con circuito variacional y QSVC con kernel de fidelidad)
y tres modelos clasicos (SVM-RBF, MLP, KNN) para clasificar el dataset Iris.  
Ademas se realizan experimentos sobre: (a) feature maps, (b) profundidad del ansatz,
(c) optimizadores, (d) sensibilidad al ruido NISQ, y (e) generalizacion en MNIST reducido.

## 1. Configuración e Importaciones

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_iris, fetch_openml
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
from sklearn.decomposition import PCA

from qiskit.circuit.library import ZZFeatureMap, ZFeatureMap, PauliFeatureMap, RealAmplitudes, EfficientSU2
from qiskit.primitives import StatevectorSampler
from qiskit_machine_learning.algorithms import VQC, QSVC
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.optimizers import COBYLA, SPSA, L_BFGS_B
from qiskit_machine_learning.utils import algorithm_globals

algorithm_globals.random_seed = 42
np.random.seed(42)

plt.rcParams.update({
    'figure.figsize': (10, 6), 'font.size': 12,
    'axes.titlesize': 14, 'axes.labelsize': 12,
})

print("✅ Todas las librerías cargadas correctamente.")

os.makedirs('figuras', exist_ok=True)


## 2. Carga y Preprocesamiento del Dataset Iris
El dataset Iris tiene 150 muestras, 4 características y 3 clases (setosa, versicolor, virginica).
Se normaliza a [0, π] para codificación angular en circuitos cuánticos.

In [ ]:
X, y = load_iris(return_X_y=True)
iris = load_iris()
feature_names = iris.feature_names
target_names = iris.target_names

# Normalizar a [0, π] para codificación angular
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_scaled = scaler.fit_transform(X)

# One-hot encoding para VQC (requiere etiquetas one-hot)
ohe = OneHotEncoder(sparse_output=False)
y_oh = ohe.fit_transform(y.reshape(-1, 1))

# Split estratificado 70/30
X_tr, X_te, y_tr_oh, y_te_oh = train_test_split(
    X_scaled, y_oh, test_size=0.3, stratify=y, random_state=42)
y_tr_lbl = y_tr_oh.argmax(axis=1)
y_te_lbl = y_te_oh.argmax(axis=1)

print(f"Entrenamiento: {X_tr.shape[0]} muestras | Test: {X_te.shape[0]} muestras")
print(f"Distribución entrenamiento: {np.bincount(y_tr_lbl)}")
print(f"Distribución test:          {np.bincount(y_te_lbl)}")

# Estadísticas descriptivas
df_iris = pd.DataFrame(X, columns=feature_names)
df_iris['clase'] = [target_names[c] for c in y]
print(f'\n📋 Estadísticas descriptivas del dataset:')
print(df_iris.describe().round(2).to_string())
print(f'\n🎯 Clases: {list(target_names)}')
print(f'📐 Dimensiones originales: {X.shape}')
print(f'📐 Rango normalizado: [0, π] ≈ [0, 3.1416]')


### 2.1 Análisis Exploratorio de Datos (EDA)

In [ ]:
# --- Scatter plots de pares de features ---
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Distribucion por pares de caracteristicas', fontsize=15, fontweight='bold')
pairs = [(0,1), (0,2), (0,3), (1,2), (1,3), (2,3)]
colors = ['#2196F3', '#FF9800', '#4CAF50']
for idx, (i, j) in enumerate(pairs):
    ax = axes[idx//3][idx%3]
    for c in range(3):
        mask = y == c
        ax.scatter(X[mask, i], X[mask, j], c=colors[c], label=target_names[c],
                   alpha=0.7, edgecolors='k', linewidth=0.5, s=50)
    ax.set_xlabel(feature_names[i], fontsize=9)
    ax.set_ylabel(feature_names[j], fontsize=9)
    if idx == 0: ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('figuras/eda_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Matriz de correlacion ---
fig, ax = plt.subplots(figsize=(7, 6))
corr = df_iris[feature_names].corr()
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels([f.replace(' (cm)', '') for f in feature_names], rotation=45, ha='right', fontsize=9)
ax.set_yticklabels([f.replace(' (cm)', '') for f in feature_names], fontsize=9)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center',
                fontsize=11, fontweight='bold',
                color='white' if abs(corr.iloc[i,j]) > 0.6 else 'black')
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('Matriz de correlacion de Pearson')
plt.tight_layout()
plt.savefig('figuras/eda_correlacion.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Boxplots por clase ---
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, ax in enumerate(axes):
    data_per_class = [X[y == c, i] for c in range(3)]
    bp = ax.boxplot(data_per_class, labels=target_names, patch_artist=True, widths=0.6)
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_title(feature_names[i].replace(' (cm)', ''), fontsize=10)
    ax.tick_params(axis='x', rotation=30)
fig.suptitle('Distribucion de cada feature por clase', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figuras/eda_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Correlacion petal length vs petal width: {corr.iloc[2,3]:.3f} (alta colinealidad)')
print(f'Setosa es linealmente separable en todos los pares de features.')


## 3. Modelos Clásicos de Referencia (Baseline)
Se implementan SVM con kernel RBF, Red Neuronal (MLP) y KNN como líneas base.

In [ ]:
resultados = {}

# --- SVM con kernel RBF ---
t0 = time.time()
svm_rbf = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm_rbf.fit(X_tr, y_tr_lbl)
t_svm = time.time() - t0
y_pred_svm = svm_rbf.predict(X_te)
acc_svm = accuracy_score(y_te_lbl, y_pred_svm)
resultados['SVM-RBF'] = {
    'accuracy': acc_svm, 'tiempo': t_svm,
    'y_pred': y_pred_svm, 'tipo': 'Clásico'
}

# --- MLP (Red Neuronal) ---
t0 = time.time()
mlp = MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=2000, random_state=42)
mlp.fit(X_tr, y_tr_lbl)
t_mlp = time.time() - t0
y_pred_mlp = mlp.predict(X_te)
acc_mlp = accuracy_score(y_te_lbl, y_pred_mlp)
resultados['MLP'] = {
    'accuracy': acc_mlp, 'tiempo': t_mlp,
    'y_pred': y_pred_mlp, 'tipo': 'Clásico'
}

# --- KNN ---
t0 = time.time()
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_tr, y_tr_lbl)
t_knn = time.time() - t0
y_pred_knn = knn.predict(X_te)
acc_knn = accuracy_score(y_te_lbl, y_pred_knn)
resultados['KNN'] = {
    'accuracy': acc_knn, 'tiempo': t_knn,
    'y_pred': y_pred_knn, 'tipo': 'Clásico'
}

print("=" * 55)
print("RESULTADOS MODELOS CLÁSICOS")
print("=" * 55)
for name, r in resultados.items():
    print(f"  {name:12s} → Accuracy: {r['accuracy']:.4f} | Tiempo: {r['tiempo']:.4f}s")

# Validación cruzada 5-fold para los modelos clásicos
print("\n📋 Validación Cruzada 5-Fold:")
for name, model in [('SVM-RBF', svm_rbf), ('MLP', mlp), ('KNN', knn)]:
    scores = cross_val_score(model, X_scaled, y, cv=5, scoring='accuracy')
    print(f"  {name:12s} → {scores.mean():.4f} ± {scores.std():.4f}")

# Reporte detallado por clase para cada modelo clásico
print('\n' + '=' * 55)
print('REPORTES DETALLADOS POR CLASE')
print('=' * 55)
for name, r in resultados.items():
    print(f'\n--- {name} ---')
    print(classification_report(y_te_lbl, r['y_pred'],
          target_names=list(target_names), zero_division=0))


## 4. Diseño del Circuito Cuántico
Visualización del feature map (ZZFeatureMap) y el ansatz (RealAmplitudes).

In [ ]:
num_qubits = 4
feature_map = ZZFeatureMap(feature_dimension=num_qubits, reps=2, entanglement='linear')
ansatz = RealAmplitudes(num_qubits=num_qubits, reps=3, entanglement='linear')
circuito_completo = feature_map.compose(ansatz)

print(f"Número de qubits: {num_qubits}")
print(f"Parámetros del feature map: {feature_map.num_parameters}")
print(f"Parámetros entrenables (ansatz): {ansatz.num_parameters}")
print(f"Profundidad del circuito completo: {circuito_completo.decompose().depth()}")

# Conteo de puertas
ops = circuito_completo.decompose().count_ops()
print(f'Puertas del circuito: {dict(ops)}')
print(f'Total de puertas: {sum(ops.values())}')

fig, axes = plt.subplots(3, 1, figsize=(16, 14))
feature_map.decompose().draw('mpl', ax=axes[0])
axes[0].set_title('Feature Map: ZZFeatureMap (reps=2, linear)', fontsize=13)
ansatz.decompose().draw('mpl', ax=axes[1])
axes[1].set_title('Ansatz: RealAmplitudes (reps=3, linear)', fontsize=13)
circuito_completo.decompose().draw('mpl', ax=axes[2])
axes[2].set_title('Circuito Completo: Feature Map + Ansatz', fontsize=13)
plt.tight_layout()
plt.savefig('figuras/circuitos.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Modelo Cuántico 1: VQC (Variational Quantum Classifier)
Clasificador cuántico variacional con ZZFeatureMap + RealAmplitudes + COBYLA.

In [ ]:
# Callback para registrar la curva de pérdida
history_vqc = {'loss': [], 'weights': []}
def callback_vqc(weights, loss, *args):
    history_vqc['loss'].append(loss)

print("🔄 Entrenando VQC (esto puede tardar ~5-10 minutos)...")
t0 = time.time()

vqc = VQC(
    feature_map=feature_map,
    ansatz=ansatz,
    loss='cross_entropy',
    optimizer=COBYLA(maxiter=200),
    sampler=StatevectorSampler(),
    callback=callback_vqc,
)
vqc.fit(X_tr, y_tr_oh)
t_vqc = time.time() - t0

acc_vqc = vqc.score(X_te, y_te_oh)
# Obtener predicciones
y_pred_raw = vqc.predict(X_te)
if len(y_pred_raw.shape) > 1:
    y_pred_vqc = y_pred_raw.argmax(axis=1)
else:
    y_pred_vqc = y_pred_raw.astype(int)

resultados['VQC'] = {
    'accuracy': acc_vqc, 'tiempo': t_vqc,
    'y_pred': y_pred_vqc, 'tipo': 'Cuántico'
}

print(f"✅ VQC entrenado en {t_vqc:.1f}s")
print(f"   Accuracy en test: {acc_vqc:.4f}")
print(f"   Iteraciones de pérdida registradas: {len(history_vqc['loss'])}")


# Reporte detallado del VQC
print(f'\n📊 Reporte detallado VQC:')
print(classification_report(y_te_lbl, y_pred_vqc,
      target_names=list(target_names), zero_division=0))
# Curva de pérdida
plt.figure(figsize=(10, 5))
plt.plot(history_vqc['loss'], color='#7B1FA2', linewidth=2)
plt.xlabel('Iteración')
plt.ylabel('Cross-Entropy Loss')
plt.title('Curva de Pérdida del VQC (COBYLA)')
plt.grid(True, alpha=0.3)
plt.savefig('figuras/vqc_loss.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Modelo Cuántico 2: QSVC (Quantum SVM con Kernel Cuántico)
Utiliza FidelityQuantumKernel con ZZFeatureMap y delega a sklearn.svm.SVC.

In [ ]:
print("🔄 Entrenando QSVC (construyendo matriz de kernel cuántico)...")
t0 = time.time()

qkernel = FidelityQuantumKernel(feature_map=feature_map)
qsvc = QSVC(quantum_kernel=qkernel)
qsvc.fit(X_tr, y_tr_lbl)
t_qsvc = time.time() - t0

y_pred_qsvc = qsvc.predict(X_te)
acc_qsvc = accuracy_score(y_te_lbl, y_pred_qsvc)

resultados['QSVC'] = {
    'accuracy': acc_qsvc, 'tiempo': t_qsvc,
    'y_pred': y_pred_qsvc, 'tipo': 'Cuántico'
}

print(f"✅ QSVC entrenado en {t_qsvc:.1f}s")
print(f"   Accuracy en test: {acc_qsvc:.4f}")

# Reporte detallado del QSVC
print(f'\n📊 Reporte detallado QSVC:')
print(classification_report(y_te_lbl, y_pred_qsvc,
      target_names=list(target_names), zero_division=0))


## 7. Comparación General: Clásico vs Cuántico

In [ ]:
print("\n" + "=" * 70)
print("TABLA COMPARATIVA: TODOS LOS MODELOS")
print("=" * 70)
print(f"{'Modelo':15s} {'Tipo':10s} {'Accuracy':>10s} {'Precision':>10s} {'Recall':>10s} {'F1':>10s} {'Tiempo':>10s}")
print("-" * 70)

for name, r in resultados.items():
    yp = r['y_pred']
    yt = y_te_lbl
    prec = precision_score(yt, yp, average='macro', zero_division=0)
    rec = recall_score(yt, yp, average='macro', zero_division=0)
    f1 = f1_score(yt, yp, average='macro', zero_division=0)
    print(f"{name:15s} {r['tipo']:10s} {r['accuracy']:10.4f} {prec:10.4f} {rec:10.4f} {f1:10.4f} {r['tiempo']:8.2f}s")


# Tabla como DataFrame para mejor visualización
tabla_data = []
for name, r in resultados.items():
    yp, yt = r['y_pred'], y_te_lbl
    tabla_data.append({
        'Modelo': name, 'Tipo': r['tipo'],
        'Accuracy': f"{r['accuracy']:.4f}",
        'Precision': f"{precision_score(yt, yp, average='macro', zero_division=0):.4f}",
        'Recall': f"{recall_score(yt, yp, average='macro', zero_division=0):.4f}",
        'F1-Score': f"{f1_score(yt, yp, average='macro', zero_division=0):.4f}",
        'Tiempo (s)': f"{r['tiempo']:.3f}",
    })
df_comp = pd.DataFrame(tabla_data)
print('\n📋 Tabla como DataFrame:')
print(df_comp.to_string(index=False))

# Gráfico de barras comparativo
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
names = list(resultados.keys())
accs = [resultados[n]['accuracy'] for n in names]
times = [resultados[n]['tiempo'] for n in names]
colors_bar = ['#2196F3' if resultados[n]['tipo']=='Clásico' else '#9C27B0' for n in names]

bars = ax1.bar(names, accs, color=colors_bar, edgecolor='white', linewidth=1.5)
ax1.set_ylabel('Accuracy')
ax1.set_title('Accuracy: Clásico vs Cuántico')
ax1.set_ylim(0, 1.1)
for bar, acc in zip(bars, accs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{acc:.1%}', ha='center', fontweight='bold')
ax1.axhline(y=0.9, color='gray', linestyle='--', alpha=0.5, label='90%')
ax1.legend()

ax2.bar(names, times, color=colors_bar, edgecolor='white', linewidth=1.5)
ax2.set_ylabel('Tiempo (s)')
ax2.set_title('Tiempo de Entrenamiento')
ax2.set_yscale('log')
plt.tight_layout()
plt.savefig('figuras/comparacion_general.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Matrices de Confusión

In [ ]:
fig, axes = plt.subplots(1, len(resultados), figsize=(5*len(resultados), 4))
if len(resultados) == 1: axes = [axes]
for idx, (name, r) in enumerate(resultados.items()):
    cm = confusion_matrix(y_te_lbl, r['y_pred'])
    im = axes[idx].imshow(cm, cmap='Purples', interpolation='nearest')
    axes[idx].set_title(f'{name}\nAcc: {r["accuracy"]:.2%}', fontweight='bold')
    axes[idx].set_xlabel('Predicción')
    axes[idx].set_ylabel('Real')
    axes[idx].set_xticks(range(3))
    axes[idx].set_yticks(range(3))
    axes[idx].set_xticklabels(target_names, rotation=45, fontsize=8)
    axes[idx].set_yticklabels(target_names, fontsize=8)
    for i in range(3):
        for j in range(3):
            axes[idx].text(j, i, str(cm[i,j]), ha='center', va='center',
                          fontweight='bold', fontsize=14,
                          color='white' if cm[i,j] > cm.max()/2 else 'black')
plt.tight_layout()
plt.savefig('figuras/matrices_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Fronteras de Decisión (Proyección PCA 2D)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_tr_2d = pca.fit_transform(X_tr)
X_te_2d = pca.transform(X_te)
x_min, x_max = X_tr_2d[:,0].min()-0.5, X_tr_2d[:,0].max()+0.5
y_min, y_max = X_tr_2d[:,1].min()-0.5, X_tr_2d[:,1].max()+0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))

# Entrenar SVM en 2D para visualizar fronteras
scaler_2d = MinMaxScaler(feature_range=(0, np.pi))
X_tr_2d_s = scaler_2d.fit_transform(X_tr_2d)
svm_2d = SVC(kernel='rbf', C=1.0, gamma='scale').fit(X_tr_2d_s, y_tr_lbl)

grid_2d = scaler_2d.transform(np.c_[xx.ravel(), yy.ravel()])
Z_svm = svm_2d.predict(grid_2d).reshape(xx.shape)

fig, ax = plt.subplots(1, 1, figsize=(10, 7))
ax.contourf(xx, yy, Z_svm, alpha=0.3, cmap='coolwarm')
for c, color, name in zip([0,1,2], ['#2196F3','#FF9800','#4CAF50'], target_names):
    mask = y_te_lbl == c
    ax.scatter(X_te_2d[mask,0], X_te_2d[mask,1], c=color, label=name,
               edgecolors='k', s=80, linewidth=0.8)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} varianza)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} varianza)')
ax.set_title('Frontera de Decisión SVM-RBF (proyección PCA 2D)')
ax.legend()
plt.savefig('figuras/frontera_decision.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Parte 1 completada. Ejecutar Parte 2 para experimentos avanzados.")

# Parte 2: Experimentos Avanzados
## Comparación de Feature Maps, Profundidad de Ansatz, Optimizadores y Ruido NISQ

**Nota:** Ejecutar después de la Parte 1 (variables compartidas).

## 10. Experimento 1: Comparación de Feature Maps
ZFeatureMap vs ZZFeatureMap (linear) vs ZZFeatureMap (full) vs PauliFeatureMap

In [ ]:
from qiskit.circuit.library import ZFeatureMap, ZZFeatureMap, PauliFeatureMap, RealAmplitudes
from qiskit.primitives import StatevectorSampler
from qiskit_machine_learning.algorithms import VQC
from qiskit_machine_learning.optimizers import COBYLA
import time, numpy as np

feature_maps = {
    'ZFeatureMap': ZFeatureMap(feature_dimension=4, reps=2),
    'ZZFeatureMap (linear)': ZZFeatureMap(feature_dimension=4, reps=2, entanglement='linear'),
    'ZZFeatureMap (full)': ZZFeatureMap(feature_dimension=4, reps=2, entanglement='full'),
    'PauliFeatureMap': PauliFeatureMap(feature_dimension=4, reps=2, paulis=['Z','ZZ']),
}

fm_results = {}
for fm_name, fm in feature_maps.items():
    print(f"🔄 Probando {fm_name}...")
    ans = RealAmplitudes(num_qubits=4, reps=3, entanglement='linear')
    hist = {'loss': []}
    def cb(w, l, *args): hist['loss'].append(l)

    t0 = time.time()
    model = VQC(
        feature_map=fm, ansatz=ans, loss='cross_entropy',
        optimizer=COBYLA(maxiter=150), sampler=StatevectorSampler(), callback=cb,
    )
    model.fit(X_tr, y_tr_oh)
    t_elapsed = time.time() - t0
    acc = model.score(X_te, y_te_oh)
    depth = fm.compose(ans).decompose().depth()

    fm_results[fm_name] = {'accuracy': acc, 'tiempo': t_elapsed, 'depth': depth, 'loss': hist['loss']}
    print(f"   ✅ {fm_name}: Acc={acc:.4f}, Depth={depth}, Tiempo={t_elapsed:.1f}s")

# Gráfico
import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

names_fm = list(fm_results.keys())
accs_fm = [fm_results[n]['accuracy'] for n in names_fm]
ax1.barh(names_fm, accs_fm, color=['#42A5F5','#7E57C2','#AB47BC','#EC407A'])
ax1.set_xlabel('Accuracy')
ax1.set_title('Comparación de Feature Maps')
ax1.set_xlim(0, 1.1)
for i, v in enumerate(accs_fm):
    ax1.text(v + 0.01, i, f'{v:.2%}', va='center', fontweight='bold')

for fm_name, r in fm_results.items():
    ax2.plot(r['loss'], label=f"{fm_name} ({r['accuracy']:.2%})", linewidth=1.5)
ax2.set_xlabel('Iteración')
ax2.set_ylabel('Loss')
ax2.set_title('Convergencia por Feature Map')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figuras/exp1_feature_maps.png', dpi=150, bbox_inches='tight')
plt.show()

# Tabla resumen Feature Maps
print('\n📋 Resumen Feature Maps:')
df_fm = pd.DataFrame([{'Feature Map': n, 'Accuracy': f"{r['accuracy']:.4f}",
    'Profundidad': r['depth'], 'Tiempo (s)': f"{r['tiempo']:.1f}"}
    for n, r in fm_results.items()])
print(df_fm.to_string(index=False))


## 11. Experimento 2: Sensibilidad a la Profundidad del Ansatz
Se varía reps ∈ {1, 2, 3, 5} en RealAmplitudes para detectar sobreajuste o barren plateau.

In [ ]:
reps_list = [1, 2, 3, 5]
depth_results = {}
best_fmap = ZZFeatureMap(feature_dimension=4, reps=2, entanglement='linear')

for reps in reps_list:
    print(f"🔄 Ansatz reps={reps}...")
    ans = RealAmplitudes(num_qubits=4, reps=reps, entanglement='linear')
    hist = {'loss': []}
    def cb(w, l, *args): hist['loss'].append(l)

    t0 = time.time()
    model = VQC(
        feature_map=best_fmap, ansatz=ans, loss='cross_entropy',
        optimizer=COBYLA(maxiter=150), sampler=StatevectorSampler(), callback=cb,
    )
    model.fit(X_tr, y_tr_oh)
    t_elapsed = time.time() - t0
    acc = model.score(X_te, y_te_oh)
    n_params = ans.num_parameters

    depth_results[reps] = {
        'accuracy': acc, 'tiempo': t_elapsed, 'n_params': n_params, 'loss': hist['loss']
    }
    print(f"   ✅ reps={reps}: Acc={acc:.4f}, Params={n_params}, Tiempo={t_elapsed:.1f}s")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

reps_vals = list(depth_results.keys())
accs_d = [depth_results[r]['accuracy'] for r in reps_vals]
params_d = [depth_results[r]['n_params'] for r in reps_vals]

ax1.plot(reps_vals, accs_d, 'o-', color='#7B1FA2', linewidth=2, markersize=10)
ax1.set_xlabel('Reps (profundidad ansatz)')
ax1.set_ylabel('Accuracy')
ax1.set_title('Accuracy vs Profundidad del Ansatz')
ax1.grid(True, alpha=0.3)
for r, a in zip(reps_vals, accs_d):
    ax1.annotate(f'{a:.2%}\n({depth_results[r]["n_params"]}p)', (r, a),
                textcoords="offset points", xytext=(0,12), ha='center', fontsize=9)

for reps, r in depth_results.items():
    ax2.plot(r['loss'], label=f'reps={reps}', linewidth=1.5)
ax2.set_xlabel('Iteración')
ax2.set_ylabel('Loss')
ax2.set_title('Convergencia por Profundidad')
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figuras/exp2_profundidad.png', dpi=150, bbox_inches='tight')
plt.show()

# Tabla resumen Profundidad
print('\n📋 Resumen Profundidad del Ansatz:')
df_depth = pd.DataFrame([{'Reps': r, 'Params': d['n_params'],
    'Accuracy': f"{d['accuracy']:.4f}", 'Tiempo (s)': f"{d['tiempo']:.1f}"}
    for r, d in depth_results.items()])
print(df_depth.to_string(index=False))


## 12. Experimento 3: Comparación de Optimizadores
COBYLA vs SPSA vs L-BFGS-B sobre la misma arquitectura VQC.

In [ ]:
optimizers = {
    'COBYLA': COBYLA(maxiter=200),
    'SPSA': SPSA(maxiter=150),
    'L-BFGS-B': L_BFGS_B(maxiter=200),
}
opt_results = {}

for opt_name, opt in optimizers.items():
    print(f"🔄 Optimizador: {opt_name}...")
    fm = ZZFeatureMap(feature_dimension=4, reps=2, entanglement='linear')
    ans = RealAmplitudes(num_qubits=4, reps=3, entanglement='linear')
    hist = {'loss': []}
    def cb(w, l, *args): hist['loss'].append(l)

    t0 = time.time()
    model = VQC(
        feature_map=fm, ansatz=ans, loss='cross_entropy',
        optimizer=opt, sampler=StatevectorSampler(), callback=cb,
    )
    model.fit(X_tr, y_tr_oh)
    t_elapsed = time.time() - t0
    acc = model.score(X_te, y_te_oh)

    opt_results[opt_name] = {
        'accuracy': acc, 'tiempo': t_elapsed,
        'iters': len(hist['loss']), 'loss': hist['loss']
    }
    print(f"   ✅ {opt_name}: Acc={acc:.4f}, Iters={len(hist['loss'])}, Tiempo={t_elapsed:.1f}s")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

opt_names = list(opt_results.keys())
accs_o = [opt_results[n]['accuracy'] for n in opt_names]
times_o = [opt_results[n]['tiempo'] for n in opt_names]

bars = ax1.bar(opt_names, accs_o, color=['#26A69A','#EF5350','#5C6BC0'], edgecolor='white')
ax1.set_ylabel('Accuracy')
ax1.set_title('Accuracy por Optimizador')
ax1.set_ylim(0, 1.1)
for bar, a in zip(bars, accs_o):
    ax1.text(bar.get_x()+bar.get_width()/2, a+0.02, f'{a:.2%}', ha='center', fontweight='bold')

for opt_name, r in opt_results.items():
    ax2.plot(r['loss'], label=f"{opt_name} ({r['accuracy']:.2%})", linewidth=2)
ax2.set_xlabel('Iteración')
ax2.set_ylabel('Loss')
ax2.set_title('Convergencia por Optimizador')
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figuras/exp3_optimizadores.png', dpi=150, bbox_inches='tight')
plt.show()

# Tabla resumen Optimizadores
print('\n📋 Resumen Optimizadores:')
df_opt = pd.DataFrame([{'Optimizador': n, 'Accuracy': f"{r['accuracy']:.4f}",
    'Iteraciones': r['iters'], 'Tiempo (s)': f"{r['tiempo']:.1f}"}
    for n, r in opt_results.items()])
print(df_opt.to_string(index=False))


## 13. Heatmap de la Matriz de Kernel Cuántico vs RBF

In [ ]:
# Subconjunto para visualización (primeras 30 muestras)
n_vis = min(30, len(X_te))
X_vis = X_te[:n_vis]

# Kernel cuántico
fm_k = ZZFeatureMap(feature_dimension=4, reps=2, entanglement='linear')
qk = FidelityQuantumKernel(feature_map=fm_k)
K_quantum = qk.evaluate(X_vis)

# Kernel RBF clásico
from sklearn.metrics.pairwise import rbf_kernel
K_rbf = rbf_kernel(X_vis, gamma=1.0/(4 * X_vis.var()))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
im1 = ax1.imshow(K_quantum, cmap='inferno', vmin=0, vmax=1)
ax1.set_title('Kernel Cuántico (ZZFeatureMap)')
plt.colorbar(im1, ax=ax1, shrink=0.8)

im2 = ax2.imshow(K_rbf, cmap='inferno', vmin=0, vmax=1)
ax2.set_title('Kernel RBF Clásico')
plt.colorbar(im2, ax=ax2, shrink=0.8)
plt.tight_layout()
plt.savefig('figuras/kernels_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. Experimento con Varianza: Boxplots sobre Múltiples Semillas

In [ ]:
print("🔄 Ejecutando VQC con 5 semillas distintas...")
seed_accuracies = []
for seed in [42, 123, 456, 789, 1024]:
    algorithm_globals.random_seed = seed
    fm = ZZFeatureMap(feature_dimension=4, reps=2, entanglement='linear')
    ans = RealAmplitudes(num_qubits=4, reps=3, entanglement='linear')
    model = VQC(
        feature_map=fm, ansatz=ans, loss='cross_entropy',
        optimizer=COBYLA(maxiter=150), sampler=StatevectorSampler(),
    )
    model.fit(X_tr, y_tr_oh)
    acc = model.score(X_te, y_te_oh)
    seed_accuracies.append(acc)
    print(f"   Semilla {seed}: {acc:.4f}")

# Restaurar semilla
algorithm_globals.random_seed = 42

svm_cv = cross_val_score(SVC(kernel='rbf', C=1.0, gamma='scale'), X_scaled, y, cv=5)

fig, ax = plt.subplots(figsize=(8, 5))
bp = ax.boxplot([seed_accuracies, svm_cv.tolist()], labels=['VQC (5 seeds)', 'SVM-RBF (5-fold CV)'],
                patch_artist=True, widths=0.5)
bp['boxes'][0].set_facecolor('#CE93D8')
bp['boxes'][1].set_facecolor('#90CAF9')
ax.set_ylabel('Accuracy')
ax.set_title('Distribución de Accuracy: VQC vs SVM-RBF')
ax.grid(True, alpha=0.3, axis='y')
plt.savefig('figuras/boxplots_varianza.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nVQC: {np.mean(seed_accuracies):.4f} ± {np.std(seed_accuracies):.4f}")
print(f"SVM: {svm_cv.mean():.4f} ± {svm_cv.std():.4f}")

## 15. Experimento Secundario: MNIST Reducido (2 clases)
Reducir MNIST a clases {0, 1}, submuestrear, PCA a 4 componentes.

In [ ]:
from sklearn.datasets import load_digits

print("🔄 Cargando MNIST reducido (digits 0 y 1)...")
digits = load_digits()
mask = np.isin(digits.target, [0, 1])
X_mnist = digits.data[mask]
y_mnist = digits.target[mask]

# PCA a 4 componentes
pca_mnist = PCA(n_components=4, random_state=42)
X_mnist_pca = pca_mnist.fit_transform(X_mnist)
X_mnist_scaled = MinMaxScaler(feature_range=(0, np.pi)).fit_transform(X_mnist_pca)

y_mnist_oh = OneHotEncoder(sparse_output=False).fit_transform(y_mnist.reshape(-1,1))
X_m_tr, X_m_te, y_m_tr, y_m_te = train_test_split(
    X_mnist_scaled, y_mnist_oh, test_size=0.3, stratify=y_mnist, random_state=42)
y_m_tr_lbl = y_m_tr.argmax(1)
y_m_te_lbl = y_m_te.argmax(1)

print(f"MNIST reducido: {X_mnist_scaled.shape[0]} muestras, {4} features (PCA)")
print(f"Train: {X_m_tr.shape[0]} | Test: {X_m_te.shape[0]}")

# SVM clásico
svm_m = SVC(kernel='rbf').fit(X_m_tr, y_m_tr_lbl)
acc_svm_m = svm_m.score(X_m_te, y_m_te_lbl)

# VQC
fm_m = ZZFeatureMap(feature_dimension=4, reps=2, entanglement='linear')
ans_m = RealAmplitudes(num_qubits=4, reps=3, entanglement='linear')
hist_m = {'loss': []}
def cb_m(w, l, *args): hist_m['loss'].append(l)

print("🔄 Entrenando VQC en MNIST reducido...")
t0 = time.time()
vqc_m = VQC(
    feature_map=fm_m, ansatz=ans_m, loss='cross_entropy',
    optimizer=COBYLA(maxiter=150), sampler=StatevectorSampler(), callback=cb_m,
)
vqc_m.fit(X_m_tr, y_m_tr)
t_m = time.time() - t0
acc_vqc_m = vqc_m.score(X_m_te, y_m_te)

print(f"\n📊 Resultados MNIST Reducido:")
print(f"   SVM-RBF:  {acc_svm_m:.4f}")
print(f"   VQC:      {acc_vqc_m:.4f} ({t_m:.1f}s)")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.bar(['SVM-RBF', 'VQC'], [acc_svm_m, acc_vqc_m], color=['#2196F3', '#9C27B0'])
ax1.set_ylabel('Accuracy')
ax1.set_title('MNIST Reducido (0 vs 1)')
ax1.set_ylim(0, 1.1)
for i, v in enumerate([acc_svm_m, acc_vqc_m]):
    ax1.text(i, v+0.02, f'{v:.2%}', ha='center', fontweight='bold')

ax2.plot(hist_m['loss'], color='#9C27B0', linewidth=2)
ax2.set_xlabel('Iteración')
ax2.set_ylabel('Loss')
ax2.set_title('Convergencia VQC en MNIST')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figuras/exp_mnist.png', dpi=150, bbox_inches='tight')
plt.show()

## 15b. Experimento: Sensibilidad al Ruido NISQ (Simulado)
En hardware cuantico real, las puertas no son perfectas. Se simula ruido
depolarizante anadiendo perturbaciones gaussianas a los datos de entrada,
lo cual emula el efecto de errores de decoherencia sobre la codificacion
de features. Se evalua como degrada el accuracy del VQC y del SVM al
incrementar el nivel de ruido.

In [ ]:
# Simulacion de ruido: perturbacion gaussiana sobre los datos de entrada
# Esto emula errores de decoherencia y puertas imperfectas en hardware NISQ
noise_levels = [0.0, 0.05, 0.10, 0.20, 0.30, 0.50]
noise_results_vqc = []
noise_results_svm = []

# Entrenar VQC limpio una vez (reusar el ya entrenado)
# Entrenar SVM limpio una vez
svm_noise = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm_noise.fit(X_tr, y_tr_lbl)

print('Evaluando sensibilidad al ruido...')
for sigma in noise_levels:
    accs_vqc_noise = []
    accs_svm_noise = []
    # Promediar sobre 5 realizaciones de ruido
    for trial in range(5):
        np.random.seed(trial)
        ruido = np.random.normal(0, sigma, X_te.shape)
        X_te_noisy = np.clip(X_te + ruido, 0, np.pi)

        # VQC
        y_pred_n = vqc.predict(X_te_noisy)
        if len(y_pred_n.shape) > 1:
            y_pred_n = y_pred_n.argmax(axis=1)
        acc_n = accuracy_score(y_te_lbl, y_pred_n.astype(int))
        accs_vqc_noise.append(acc_n)

        # SVM
        acc_s = accuracy_score(y_te_lbl, svm_noise.predict(X_te_noisy))
        accs_svm_noise.append(acc_s)

    noise_results_vqc.append((np.mean(accs_vqc_noise), np.std(accs_vqc_noise)))
    noise_results_svm.append((np.mean(accs_svm_noise), np.std(accs_svm_noise)))
    print(f'  sigma={sigma:.2f} -> VQC: {np.mean(accs_vqc_noise):.4f} +/- {np.std(accs_vqc_noise):.4f}'
          f'  |  SVM: {np.mean(accs_svm_noise):.4f} +/- {np.std(accs_svm_noise):.4f}')

# Restaurar semilla
np.random.seed(42)

# Grafico
fig, ax = plt.subplots(figsize=(10, 6))
vqc_means = [r[0] for r in noise_results_vqc]
vqc_stds  = [r[1] for r in noise_results_vqc]
svm_means = [r[0] for r in noise_results_svm]
svm_stds  = [r[1] for r in noise_results_svm]

ax.errorbar(noise_levels, vqc_means, yerr=vqc_stds, fmt='o-',
            color='#9C27B0', linewidth=2, markersize=8, capsize=4, label='VQC')
ax.errorbar(noise_levels, svm_means, yerr=svm_stds, fmt='s-',
            color='#2196F3', linewidth=2, markersize=8, capsize=4, label='SVM-RBF')
ax.set_xlabel('Nivel de ruido (sigma)', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Degradacion del Accuracy bajo Ruido NISQ Simulado', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig('figuras/exp_ruido_nisq.png', dpi=150, bbox_inches='tight')
plt.show()

# Tabla
print('\nResumen de sensibilidad al ruido:')
df_noise = pd.DataFrame({
    'Sigma': noise_levels,
    'VQC Acc': [f'{m:.4f} +/- {s:.4f}' for m, s in noise_results_vqc],
    'SVM Acc': [f'{m:.4f} +/- {s:.4f}' for m, s in noise_results_svm],
})
print(df_noise.to_string(index=False))
print(f'\nCaida VQC (sigma=0 vs 0.5): '
      f'{noise_results_vqc[0][0] - noise_results_vqc[-1][0]:.4f}')
print(f'Caida SVM (sigma=0 vs 0.5): '
      f'{noise_results_svm[0][0] - noise_results_svm[-1][0]:.4f}')


## 17. Tabla Resumen Final

In [ ]:
print("\n" + "=" * 80)
print("RESUMEN FINAL DE TODOS LOS EXPERIMENTOS")
print("=" * 80)

print("\n📌 Experimento Principal (Iris):")
for name, r in resultados.items():
    print(f"   {name:15s} → Accuracy: {r['accuracy']:.4f} | Tiempo: {r['tiempo']:.2f}s")

print(f"\n📌 Comparación Feature Maps:")
for name, r in fm_results.items():
    print(f"   {name:25s} → Acc: {r['accuracy']:.4f} | Depth: {r['depth']}")

print(f"\n📌 Profundidad Ansatz:")
for reps, r in depth_results.items():
    print(f"   reps={reps} ({r['n_params']:2d} params) → Acc: {r['accuracy']:.4f}")

print(f"\n📌 Optimizadores:")
for name, r in opt_results.items():
    print(f"   {name:12s} → Acc: {r['accuracy']:.4f} | Iters: {r['iters']} | Tiempo: {r['tiempo']:.1f}s")

print(f"\n📌 MNIST Reducido:")
print(f"   SVM-RBF: {acc_svm_m:.4f} | VQC: {acc_vqc_m:.4f}")

print(f"\n📌 Varianza VQC (5 seeds): {np.mean(seed_accuracies):.4f} ± {np.std(seed_accuracies):.4f}")

# Resumen consolidado como DataFrame
print('\n' + '=' * 80)
print('TABLA CONSOLIDADA DE TODOS LOS RESULTADOS')
print('=' * 80)
all_results = []
for name, r in resultados.items():
    yp, yt = r['y_pred'], y_te_lbl
    all_results.append({'Modelo': name, 'Dataset': 'Iris',
        'Tipo': r['tipo'],
        'Accuracy': f"{r['accuracy']:.4f}",
        'F1-macro': f"{f1_score(yt, yp, average='macro', zero_division=0):.4f}",
        'Tiempo': f"{r['tiempo']:.2f}s"})
all_results.append({'Modelo': 'SVM-RBF', 'Dataset': 'MNIST-01',
    'Tipo': 'Clásico', 'Accuracy': f'{acc_svm_m:.4f}',
    'F1-macro': '-', 'Tiempo': '-'})
all_results.append({'Modelo': 'VQC', 'Dataset': 'MNIST-01',
    'Tipo': 'Cuántico', 'Accuracy': f'{acc_vqc_m:.4f}',
    'F1-macro': '-', 'Tiempo': f'{t_m:.2f}s'})
df_all = pd.DataFrame(all_results)
print(df_all.to_string(index=False))
print(f'\n📊 Varianza VQC (5 seeds): μ={np.mean(seed_accuracies):.4f}, σ={np.std(seed_accuracies):.4f}')
print(f'📊 Varianza SVM (5-fold):  μ={svm_cv.mean():.4f}, σ={svm_cv.std():.4f}')

print(f'\nResultados de Ruido NISQ:')
print(f'  Sin ruido  -> VQC: {noise_results_vqc[0][0]:.4f} | SVM: {noise_results_svm[0][0]:.4f}')
print(f'  sigma=0.20 -> VQC: {noise_results_vqc[3][0]:.4f} | SVM: {noise_results_svm[3][0]:.4f}')
print(f'  sigma=0.50 -> VQC: {noise_results_vqc[5][0]:.4f} | SVM: {noise_results_svm[5][0]:.4f}')


## 18. Conclusiones y Discusion

### Resultados obtenidos

En el dataset Iris, los modelos clasicos (SVM-RBF, MLP, KNN) obtuvieron un accuracy
superior al de los modelos cuanticos (VQC, QSVC). Este resultado es consistente con la
literatura existente, dado que Iris es un problema de baja dimensionalidad (4 features)
con clases parcialmente separables de forma lineal, donde los clasificadores clasicos
ya alcanzan rendimiento cercano al optimo.

### Observaciones experimentales

- **Feature maps**: ZZFeatureMap con entrelazamiento lineal ofrecio el mejor balance
  entre profundidad de circuito y accuracy. ZFeatureMap (sin entrelazamiento) resulto
  insuficiente para capturar correlaciones entre features.
- **Profundidad del ansatz**: Se observo un punto optimo en reps=3. Con reps=5 el
  accuracy no mejoro, posiblemente por efecto de barren plateau.
- **Optimizadores**: COBYLA fue el mas estable para simulacion statevector.
  SPSA mostro mayor varianza pero es el recomendado para hardware real.
- **Ruido NISQ**: La perturbacion gaussiana degrado ambos modelos, pero el VQC
  mostro mayor sensibilidad dado que el ruido afecta directamente la codificacion
  angular en las puertas de rotacion.
- **MNIST reducido**: Ambos modelos alcanzaron alto accuracy en la tarea binaria
  (digitos 0 vs 1), confirmando que el VQC puede funcionar en problemas bien
  separables con PCA.

### Limitaciones del enfoque cuantico actual

1. El overhead computacional del VQC es significativamente mayor que el de SVM.
2. Con solo 4 qubits y simulador statevector, no se puede demostrar ventaja cuantica.
3. La varianza entre semillas del VQC indica sensibilidad a la inicializacion.

### Cuando podria QML superar a ML clasico

Segun Huang et al. (2021) y Liu et al. (2021), las ventajas potenciales del QML
aparecen cuando los datos provienen de un proceso inherentemente cuantico
(e.g., simulacion molecular), o cuando el espacio de hipotesis requerido crece
exponencialmente con la dimension del problema.

### Referencias
- Biamonte et al. (2017). Quantum machine learning. *Nature*, 549, 195-202.
- Havlicek et al. (2019). Supervised learning with quantum-enhanced feature spaces. *Nature*, 567, 209-212.
- Cerezo et al. (2021). Variational quantum algorithms. *Nat. Rev. Phys.*, 3, 625-644.
- Huang et al. (2021). Power of data in quantum machine learning. *Nat. Commun.*, 12, 2631.
- Larocca et al. (2025). A review of quantum machine learning. *Nat. Rev. Phys.*


In [ ]:
print("\n🎓 Notebook completado exitosamente.")
print("📁 Todas las figuras guardadas en ./figuras/")